# Random Forest for Handwritten Thai Digits 51–55

Same dataset, same evaluation, different model. Where the CNN notebook learns convolutional features, this notebook treats each 28×28 image as a 784-dim pixel vector and trains a `RandomForestClassifier` on it. Saves to `model.joblib`.

Workflow:
1. Upload dataset
2. Load images → flat pixel vectors
3. Offline augmentation (multiply training set by ~10×)
4. Fit Random Forest
5. Evaluate (accuracy, precision, recall, F1, confusion matrix)
6. Save + feature-importance heatmap

In [ ]:
!pip -q install scikit-learn joblib pillow

## 1. Upload / locate the dataset

**Option A — upload `dataset.zip`:**

In [ ]:
from google.colab import files
import zipfile, os

uploaded = files.upload()
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    z.extractall('.')

DATA_DIR = 'dataset'
print('classes found:', sorted(os.listdir(DATA_DIR)))

**Option B — Drive mount** (skip Option A if you use this):

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = '/content/drive/MyDrive/dataset'

## 2. Imports + config

In [ ]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from collections import Counter

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
)

from torchvision import transforms     # reused only for augmentation transforms
import joblib

IMG_SIZE      = 28
VAL_SPLIT     = 0.15
AUG_PER_IMAGE = 10        # offline copies per original training image
SEED          = 42
np.random.seed(SEED)

## 3. Load images → flat pixel vectors

Each 28×28 grayscale image becomes a 784-d vector. Pixel values stay as `uint8` (0–255) — Random Forest is scale-invariant, no normalization needed.

In [ ]:
class_names = sorted(d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)))
assert class_names == ['51','52','53','54','55'], class_names
label_to_idx = {c: i for i, c in enumerate(class_names)}
print('classes:', class_names)

samples = []   # list of (PIL.Image, int_label)
for c in class_names:
    for path in glob.glob(os.path.join(DATA_DIR, c, '*')):
        try:
            img = Image.open(path).convert('L').resize((IMG_SIZE, IMG_SIZE))
            samples.append((img, label_to_idx[c]))
        except Exception as e:
            print('skipped', path, e)

print(f'total samples: {len(samples)}')
print('per-class counts:', {class_names[i]: c for i, c in Counter(y for _, y in samples).items()})

### Sanity-check: one original sample per class

In [ ]:
fig, axes = plt.subplots(1, len(class_names), figsize=(2*len(class_names), 2.2))
for ax, cls in zip(axes, class_names):
    img, _ = next(s for s in samples if class_names[s[1]] == cls)
    ax.imshow(img, cmap='gray', vmin=0, vmax=255); ax.set_title(cls); ax.axis('off')
plt.tight_layout(); plt.show()

## 4. Offline augmentation

RF doesn't have epochs, so we can't apply random aug *during* training. Instead, we materialize `AUG_PER_IMAGE` augmented copies of each image and add them to the training set. With ~500 originals × 10 = ~5000 training samples after augmentation.

In [ ]:
aug = transforms.Compose([
    transforms.RandomAffine(
        degrees=15, translate=(0.12, 0.12), scale=(0.75, 1.25), shear=8, fill=255,
    ),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.4, fill=255),
    transforms.RandomApply([transforms.ElasticTransform(alpha=20.0, sigma=4.0)], p=0.25),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.5, 1.2))], p=0.2),
])

def to_vec(pil_img):
    return np.asarray(pil_img, dtype=np.uint8).reshape(-1)   # 784-d

X_list, y_list = [], []
for img, label in samples:
    X_list.append(to_vec(img))      # original
    y_list.append(label)
    for _ in range(AUG_PER_IMAGE):
        X_list.append(to_vec(aug(img)))
        y_list.append(label)

X = np.stack(X_list).astype(np.uint8)
y = np.asarray(y_list, dtype=np.int64)
print('X:', X.shape, X.dtype, '   y:', y.shape, y.dtype)
print('after-aug per-class:', {class_names[i]: c for i, c in sorted(Counter(y).items())})

### Sanity-check: augmented samples

Each row should still be readable as the right class — if augmentation is too strong they'll be unreadable, if too weak they'll all look identical.

In [ ]:
N_VARIANTS = 8
fig, axes = plt.subplots(len(class_names), N_VARIANTS, figsize=(N_VARIANTS * 1.2, len(class_names) * 1.3))
for row, cls in enumerate(class_names):
    img, _ = next(s for s in samples if class_names[s[1]] == cls)
    for col in range(N_VARIANTS):
        axes[row, col].imshow(aug(img), cmap='gray', vmin=0, vmax=255)
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(cls, rotation=0, labelpad=18, fontsize=12)
plt.tight_layout(); plt.show()

## 5. Train/val split

Stratified so each class is split proportionally.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=VAL_SPLIT, stratify=y, random_state=SEED,
)
print(f'train: {len(X_train)}   val: {len(X_val)}')

## 6. Train Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=400,
    max_depth=None,        # let trees grow until pure / min_samples_leaf
    min_samples_leaf=1,
    max_features='sqrt',   # standard RF default for classification
    n_jobs=-1,
    class_weight='balanced',
    random_state=SEED,
    verbose=0,
)
%time rf.fit(X_train, y_train)
print('done.')

## 7. Evaluate: accuracy / precision / recall / F1 / confusion matrix

In [ ]:
y_pred = rf.predict(X_val)

acc        = accuracy_score(y_val, y_pred)
prec_macro = precision_score(y_val, y_pred, average='macro', zero_division=0)
rec_macro  = recall_score(y_val, y_pred,    average='macro', zero_division=0)
f1_macro   = f1_score(y_val, y_pred,        average='macro', zero_division=0)

print(f'Accuracy : {acc:.4f}')
print(f'Precision: {prec_macro:.4f}  (macro)')
print(f'Recall   : {rec_macro:.4f}  (macro)')
print(f'F1-score : {f1_macro:.4f}  (macro)\n')

print('Per-class report:')
print(classification_report(y_val, y_pred, target_names=class_names, zero_division=0, digits=4))

cm = confusion_matrix(y_val, y_pred)
print('Confusion matrix (rows = true, cols = pred):')
print('       ' + '  '.join(f'{c:>4}' for c in class_names))
for i, row in enumerate(cm):
    print(f'  {class_names[i]:>3}  ' + '  '.join(f'{v:>4d}' for v in row))

## 8. Save model as `.joblib`

We bundle the model with metadata (class names, image shape, normalization expectations) so the inference side knows how to feed it correctly.

In [ ]:
bundle = {
    'model':       rf,
    'classes':     class_names,
    'image_shape': (IMG_SIZE, IMG_SIZE),
    'input':       'flatten 28x28 grayscale uint8 to 784-d vector — no normalization',
}
joblib.dump(bundle, 'model.joblib', compress=3)
print('saved model.joblib —', os.path.getsize('model.joblib') / 1024, 'KB')

from google.colab import files as gfiles
gfiles.download('model.joblib')

## 9. Quick sanity check

In [ ]:
loaded = joblib.load('model.joblib')
rf2    = loaded['model']
names  = loaded['classes']

idx  = np.random.choice(len(X_val), size=12, replace=False)
pred = rf2.predict(X_val[idx])
true = y_val[idx]
print('true:', [names[i] for i in true])
print('pred:', [names[i] for i in pred])

## 10. Feature-importance heatmap (RF interpretability bonus)

Random Forests can tell you which input pixels mattered most across all trees. Reshaped back to 28×28 it shows where the model "looks" — usually concentrated where the digit strokes typically are.

In [ ]:
imp = rf.feature_importances_.reshape(IMG_SIZE, IMG_SIZE)
fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(imp, cmap='hot')
ax.set_title('Pixel importance across all trees')
ax.axis('off')
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.tight_layout(); plt.show()